# Global CDR calculations
This script walks through how we scale up each region's representative run to calculate global CDR outcomes.

**Note:** This same workflow works for the `constApp` cases where cropland is capped at 100 percent and application evolves to track it, and for the cases where it has no cap at all. That's because the cap is imposed by the mass of existing deployments exceeding the mass of GCAM application -- a condition that only emerges in the simulations where we have to increase the rock application to mitigate overshoot of the 100 percent cropland condition. If rock application is truly constant (no overshoot response) then the cap never comes into play.

There are a few different ways we can scale up rock application. All would require the total amount of rock spread in each region to reach whatever GCAM simulated. This script assumes a fixed application rate and solves for the area required to reach the GCAM target.

The representative run's application amount was constructed with the following rules:
1. Assume a fixed application flux for each region (in t/ha/yr) and solve for the area required to match the GCAM amount of rock for that region and year.
2. If the cropland area required exceeds the total area available, increase the application flux accordingly (holding the area constant at 100%).

## Postprocessing in a single region
Since more area can be added to the region's ERW over time, we need to start the new area from "square 1" (t=0). That way ERW that starts in 2040 doesn't implicitly carry pH buffering neutralization that's been ongoing since 2020.

We do this by creating one new "deployment" (or "application cohort") every year (some deployments will spread no rock if total application doesn't increase). Each deployment follows the CDR trajectory of the region's representative run and is assigned an area equal to the change in regional area available for ERW. We ensure that the total amount of rock spread in the region each year aligns with the GCAM requirement.

### Year = 0 calculation
We start with the year 0 condition, letting $j$ refer to year and $i$ to a deployment. Since we only have one deployment in year 0, we spread all of the rock GCAM prescribes to that region and year to a single deployment. Formally,

$$M_\text{D, i} = M_\text{gcam, j}$$

when $i$ and $j$ equal zero. We can then calculate the total area required for the year 0 deployment ($A_\text{D, i}$; ha) because we know the application flux ($R_\text{j}$; t/ha/yr) in year 0 (and every year):

$$A_{D, 0} = \frac{M_\text{D, 0}}{R_0}.$$

Similarly, CDR for year 0 is just the CDR for that single deployment.

### Year > 0 calculation
For each year after year 0, the new annual deployment depends on how much *new* rock is spread relative to what existing deployments are already going to do. For each deployment, $i$, we calculate the mass of rock spread in a year by:

$$M_\text{D, i, j} = R_\text{j} \times A_\text{D, i}$$

Then, we get the amount of rock spread by existing deployments ($\mu_\text{j}$; t/yr) by summing them:

$$\mu_\text{j} = \sum_\text{i=0}^n M_\text{D, i, j}$$

Next, we can calculate how much new rock needs to be spread in year $j$ by comparing what existing deployments will do to what GCAM requires ($M_\text{GCAM, j}$; t/yr):

$$M_\text{new, j} = M_\text{GCAM, j} - \mu_\text{j}.$$

This new rock mass will be spread across the year's new deployment, at an area determined by the deployment's application flux that year. Since new deployments always start at $R_0$, we get:

$$A_\text{D, i} = \frac{M_\text{new, j}}{R_0}.$$

Deployment areas are generally fixed through time, with one exception. Since we rely on a single representative run to simulate many ERW deployments evolving over time, it's possible to reach a condition where $M_\text{new, j}$ is negative. That's because new deployments always start at $R_0$, but the representative rock spreading timeseries was calculating implicitly assuming new area kicks in at $R_j$, which can exceed $R_0$ in regions where there isn't enough cropland to hit $M_\text{GCAM, j}$ if $R_j=R_0$. The lower $R_0$ of the new deployment, compared to $R_j$, leads to a rock deficit that needs to be made up by excess area. When that deployment finally hits $R_j$ later on, it'll have "too much" area and we might see $M_\text{new, j}$ become negative.

There's no clear physical meaning for a negative $M_\text{new, j}$. It's just a scaling quirk that we have to deal with that falls out of our decision to use a single representative run to capture the time-evolution of many emerging "deployments" through time. We expect that quirk to be pretty small, likely much smaller than the uncertainty associated with other model decisions like the specific surface area or CEC. Moreover, as long as GCAM deployments see rock amendments stabilizing for long enough after scale-up (and most do), deployment areas, too, will stabilize back so the total area equals the available cropland area.

Since the more recent deployments are typically responsible for the area (and, later, mass) overshoot, we trim area off those first (first in, first out). If they need to be totally trimmed, then they are "turned off" completely. If $M_\text{new, j}$ becomes positive later, we start with a new deployment rather than turning a trimmed one back on or increasing its area.

We could alternatively explore a last in, first out rule, or a "dormancy" rule where trimmed deployments can come back (picking up where they left off) when $M_\text{new, j}$ turns positive. Generally, we don't expect $M_\text{new, j}$ to become positive again after it first turns negative since all individual deployments see their $R$ staying the same or increasing monotonically.

### Calculating CDR
The area calculations are the tricky part. Once we have those, calculating CDR for a given region and year ($\text{CDR}_\text{j}$; t/yr) is simple. It's just the sum of every deployment's yearly CDR flux (t/ha/yr) multiplied by the deployment's area:

$$\text{CDR}_\text{j} = \sum_{i=0}^n \text{CDR}_\text{rate, i, j} \times A_\text{D, i}$$

### Variables
This postprocessing effort produces or uses the following variables, defined over the following coordinates:
| VAR                   | COORDS                     | DESCRIPTION                    | UNITS          |
|-----------------------|----------------------------|--------------------------------|----------------|
| $R$                   | time, region, deployment   | application flux               | t/ha/yr        |
| $A_\text{D}$          | time, region, deployment   | deployment area                | ha             |
| $A_\text{total, ERW}$ | time, region               | total area for ERW             | ha             |
| $M_\text{GCAM}$       | time, region               | total rock spread              | t              |
| $M_\text{D}$          | time, region, deployment   | rock spread in deployment      | t              |
| $\mu$                 | time, region               | rock spread in old deployments | t              |
| $M_\text{new}$        | time, region               | rock spread in new deployment  | t              |
| $CDR_\text{flux}$     | time, region, deployment   | deployment-specific CDR        | t/ha/yr        |
| $CDR_\text{rate}$     | time, region               | region-specific CDR            | t/yr           |
| $A_\text{frac}$       | time, region               | total ERW area over total cropland area | ha    |


In [18]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

import os

# --- read in cdr calculation functions
from ew_workflows import cdr_fxns_postproc as cfp
from ew_workflows.cflx_proc import scepter_rco2
# ---
#
# ----------------------------------------------------------------
# [ FOCUS RUNS ]
# exe_central1p5 ; exe_techBehaviorChange1p5 ; exe_noWeathering1p5
# ----------------------------------------------------------------
#
# ************************************************
# --- [ SCEPTER OUTPUTS ]
EXP_GROUP = "gcam_opr_cecdr"
EXP_SET = 'exe_central1p5'
SAVE_GROUP = "gcam"
idx = "001"
DUSTSP = 'gbas'   # just for RCO2 determination
#
# --- read in time-transient and integrated data
datpath = f"s3://carbonplan-carbon-removal/ew-workflows-data/scepter/postprocessed/{SAVE_GROUP}/{EXP_GROUP}/{EXP_SET}_{idx}"
fn_cdr = "ds_cdr.zarr"                  # time-integrated data
fn_cdr_trans = "ds_cdr_transient.zarr"  # time-transient data
# read in
ds = xr.open_zarr(os.path.join(datpath, fn_cdr))
ds_trans = xr.open_zarr(os.path.join(datpath, fn_cdr_trans))

# DGSA collapses apprate_base (etc.) into "collapsed_inputs", so promote it to a coord
IS_DGSA = 'apprate_base' not in ds.coords
if IS_DGSA:
    ds = ds.set_coords('apprate_base')
    ds_trans = ds_trans.set_coords('apprate_base')

# extract apprates for reading in rock flux data
apprates_arr = np.unique(ds.apprate_base.values)

# ************************************************
# --- [ ROCK FLUX INPUTS ]
RUN_FORMAT = 'one-cell-per-region'
DEG_RES = 1
rock_fn = f'annual_rockflx_densest-ag-region_{DEG_RES}deg.pkl'
ag_fn = f'cropland+gcamRegions_time_{DEG_RES}deg.zarr'

if IS_DGSA:
    # GCAM rock target and cropland map are apprate-invariant, so reuse the
    # reference pkl/zarr and rescale apprate-dependent columns analytically
    REF_APPRATE = 5.0  # build_dgsa_climatedirs' default
    ref_apprate_name = f'{REF_APPRATE}-tHaYr-appRate-FIXED'
    ref_gcam_dir = f's3://carbonplan-carbon-removal/ew-workflows-data/gcam/apprate_calculated/{RUN_FORMAT}/{EXP_SET}/{ref_apprate_name}'
    df_rock_ref = pd.read_pickle(os.path.join(ref_gcam_dir, rock_fn))
    dsag_ref = xr.open_zarr(os.path.join(ref_gcam_dir, ag_fn))

    dfrock_dict = {}
    dsag_dict = {}
    for apprate in apprates_arr:
        apprate_name = f'{float(apprate)}-tHaYr-appRate-FIXED'
        scale = float(apprate) / REF_APPRATE
        df_this = df_rock_ref.copy()
        # FIXED apprate: actual == effective rate, other columns scale linearly
        # (calculate-rock-fluxes-from-GCAM.ipynb Scenario 3)
        df_this['effective_rate_t_ha'] = float(apprate)
        df_this['rockflx_t_ha'] = float(apprate)
        df_this['rockflx_Mt_ha'] = df_this['rockflx_t_ha'] / 1e6
        df_this['rockflx_Mt'] = df_rock_ref['rockflx_Mt'] * scale
        df_this['cropland_area_required_ha'] = df_rock_ref['cropland_area_required_ha'] / scale
        df_this['cropland_fraction_required'] = df_rock_ref['cropland_fraction_required'] / scale
        df_this['cropland_fraction_effective'] = df_this['cropland_fraction_required']
        dfrock_dict[apprate_name] = df_this
        dsag_dict[apprate_name] = dsag_ref
else:
    dfrock_dict = {}
    for apprate in apprates_arr:
        apprate_name = f'{apprate}-tHaYr-appRate-FIXED'
        gcam_dir = f's3://carbonplan-carbon-removal/ew-workflows-data/gcam/apprate_calculated/{RUN_FORMAT}/{EXP_SET}/{apprate_name}'
        # [ read in ]
        dfrock_dict[apprate_name] = pd.read_pickle(os.path.join(gcam_dir, rock_fn))

    # ************************************************
    # --- [ CROPLAND MAP ]
    # ( only used for crop areas so we just need one map )
    dsag_dict = {}
    for apprate in apprates_arr:
        apprate_name = f'{apprate}-tHaYr-appRate-FIXED'
        gcam_dir = f's3://carbonplan-carbon-removal/ew-workflows-data/gcam/apprate_calculated/{RUN_FORMAT}/{EXP_SET}/{apprate_name}'
        # [ read in ]
        dsag_dict[apprate_name] = xr.open_zarr(os.path.join(gcam_dir, ag_fn))

## Set up postprocessing decisions

In [20]:
# ---- cdr var used for global calculations
cdr_var     = 'cdr_adv'
# ---- skip the initial numerical blip (known SCEPTER quirk) ----
t_start = 0.2        # [yr] blip resolves by ~0.1
# ---- set up the rco2 correction factor
RCO2_GCAM = 0.3      # [tCDR/tRock] cdr potential under perfect efficiency
RCO2_SCEPTER = scepter_rco2[DUSTSP]  # [tCDR/tRock] cdr potential under perfect efficiency
rco2_correction_factor = RCO2_GCAM / RCO2_SCEPTER
# ---- which data to start with (time-integrated or transient)
USE_INTEGRATED = True   # derive fluxes from integrated data (avoids transient aliasing)

In [21]:
# varlist_deploy          : (time, deployment) on the annual grid
# varlist_region          : area-scaled regional total
# varist_region_cumfrac   : cumulative rock-weighted regional mean (fractions)
# varlist_rco2_correction : scaled to GCAM's RCO2

_catbudget_vars = [
    'catbudget_gbas', 'catbudget_adv', 'catbudget_tflx',
    'catbudget_sic', 'catbudget_other', 'catbudget_residual',
]

varlist_deploy      = [
    'cdr_adv', 'cdr_dif', 'cdr_dif_component', 'cdr_resp_component', 'cdr_resp_component_noAE',
    'cdr_SIConly', 'co2pot_adv_tonHaYr_camg', 'co2pot_adv_tonHaYr_totcat', 'co2pot_tot_tonHaYr_camg',
    'co2pot_tot_tonHaYr_totcat', 'DICpot_adv_camg', 'DICpot_adv_totcat', 'fraction_remaining_dissolved',
    'fraction_sld_advected', 'fraction_sld_remaining', 'fraction_total_dissolved',
    'int_dust_ton_ha_yr', 'noncarbsld_source_ca', 'noncarbsld_source_mg', 'noncarbsld_source_k',
    'noncarbsld_source_na', 'total_dissolution',
] + _catbudget_vars

varlist_region      = [
    'cdr_adv', 'cdr_dif', 'cdr_dif_component', 'cdr_resp_component', 'cdr_resp_component_noAE',
    'cdr_SIConly', 'co2pot_adv_tonHaYr_camg', 'co2pot_adv_tonHaYr_totcat', 'co2pot_tot_tonHaYr_camg',
    'co2pot_tot_tonHaYr_totcat',
] + _catbudget_vars

varist_region_cumfrac = [
    'fraction_remaining_dissolved',
    'fraction_sld_advected', 'fraction_sld_remaining', 'fraction_total_dissolved',
]
# catbudget_* are not fractions

varlist_rco2_correction = [
    'cdr_adv', 'cdr_dif', 'cdr_dif_component', 'cdr_resp_component', 'cdr_resp_component_noAE',
    'cdr_SIConly', 'co2pot_adv_tonHaYr_camg', 'co2pot_adv_tonHaYr_totcat', 'co2pot_tot_tonHaYr_camg',
    'co2pot_tot_tonHaYr_totcat', 'DICpot_adv_camg', 'DICpot_adv_totcat', 'noncarbsld_source_ca',
    'noncarbsld_source_mg', 'noncarbsld_source_k', 'noncarbsld_source_na',
] + _catbudget_vars
# catbudget terms scale with RCO2, so correct them too

## Run the postprocess

In [22]:
# --- postprocess every ds_trans combo (regional + global CDR)

ds_all = cfp.gcam_postprocess_all_dims_constApp(
    ds_trans                = ds_trans,
    dfrock_dict             = dfrock_dict,
    dsag_dict               = dsag_dict,
    varlist_deploy          = varlist_deploy,
    varlist_region          = varlist_region,
    varlist_region_cumfrac  = varist_region_cumfrac,
    varlist_rco2_correction = varlist_rco2_correction,
    rco2_correction_factor  = rco2_correction_factor,
    cdr_var                 = cdr_var,
    t_start                 = t_start,
    use_integrated          = USE_INTEGRATED,
    ds                      = ds,   # only used if `USE_INTEGRATED = True`
    apprate_key_fn          = lambda v: f'{float(v)}-tHaYr-appRate-FIXED',
)


print('\nActive sites:', list(ds_all.site.values))
print('Output dims:', dict(ds_all.sizes))


ds_all

[1/700]  site=Brazil, apprate_base=10.0, dustrad=0.25, cec_factor=0.1
[2/700]  site=Brazil, apprate_base=10.0, dustrad=0.25, cec_factor=0.5333333333333333
[3/700]  site=Brazil, apprate_base=10.0, dustrad=0.25, cec_factor=0.9666666666666668
[4/700]  site=Brazil, apprate_base=10.0, dustrad=0.25, cec_factor=1.4
[5/700]  site=Brazil, apprate_base=10.0, dustrad=0.25, cec_factor=1.8333333333333337
[6/700]  site=Brazil, apprate_base=10.0, dustrad=0.25, cec_factor=2.266666666666667
[7/700]  site=Brazil, apprate_base=10.0, dustrad=0.25, cec_factor=2.7
[8/700]  site=Brazil, apprate_base=10.0, dustrad=0.25, cec_factor=3.1333333333333333
[9/700]  site=Brazil, apprate_base=10.0, dustrad=0.25, cec_factor=3.566666666666667
[10/700]  site=Brazil, apprate_base=10.0, dustrad=0.25, cec_factor=4.0
[11/700]  site=Brazil, apprate_base=10.0, dustrad=11.333333333333334, cec_factor=0.1
[12/700]  site=Brazil, apprate_base=10.0, dustrad=11.333333333333334, cec_factor=0.5333333333333333
[13/700]  site=Brazil, app

<xarray.Dataset> Size: 1GB
Dimensions:                                     (site: 7, apprate_base: 1,
                                                 dustrad: 10, cec_factor: 10,
                                                 time: 80, deployment: 80)
Coordinates:
  * site                                        (site) <U14 392B 'Brazil' ......
  * apprate_base                                (apprate_base) float64 8B 10.0
  * dustrad                                     (dustrad) float64 80B 0.25 .....
  * cec_factor                                  (cec_factor) float64 80B 0.1 ...
  * time                                        (time) float64 640B 0.0 ... 79.0
  * deployment                                  (deployment) int64 640B 0 ... 79
Data variables: (12/65)
    cdr_adv_deployment                          (site, apprate_base, dustrad, cec_factor, time, deployment) float64 36MB ...
    rockflx_t_ha_yr                             (site, apprate_base, dustrad, cec_factor, time, deployment) float64 36MB ...
    rco2_correction_factor                      (site, apprate_base, dustrad, cec_factor) float64 6kB ...
    cdr_dif_deployment                          (site, apprate_base, dustrad, cec_factor, time, deployment) float64 36MB ...
    cdr_dif_component_deployment                (site, apprate_base, dustrad, cec_factor, time, deployment) float64 36MB ...
    cdr_resp_component_deployment               (site, apprate_base, dustrad, cec_factor, time, deployment) float64 36MB ...
    ...                                          ...
    total_dissolution_region                    (site, apprate_base, dustrad, cec_factor, time) float64 448kB ...
    int_dust_ton_ha_yr_region                   (site, apprate_base, dustrad, cec_factor, time) float64 448kB ...
    fraction_total_dissolved_fromIntFlx_region  (site, apprate_base, dustrad, cec_factor, time) float64 448kB ...
    gcam_cdr_globe                              (apprate_base, dustrad, cec_factor, time) float64 64kB ...
    cdr_adv_globe                               (apprate_base, dustrad, cec_factor, time) float64 64kB ...
    M_D_region                                  (site, apprate_base, dustrad, cec_factor, time) float64 448kB ...
Attributes:
    region:   Brazil

In [23]:
# --- reattach swept DGSA parameters as plain variables (not coordinates)
# names parsed from the "col1=val1 | col2=val2 | ..." collapsed_inputs string
if IS_DGSA:
    sample_combo = str(ds['collapsed_inputs'].values[0])
    param_cols = [kv.split('=')[0].strip() for kv in sample_combo.split('|')]
    param_cols = [c for c in param_cols if c in ds.variables]
    ds_params = ds[param_cols].reset_coords()  # demote e.g. apprate_base back to a variable
    # strip zarr v3 encoding (breaks the zarr_format=2 save)
    ds_params = ds_params.drop_encoding()
    ds_all = ds_all.merge(ds_params, join='left')

# Separate out just the regional / global data
Just to save a smaller version of the file

In [24]:
ds_sum = ds_all.drop_dims("deployment")
ds_sum

<xarray.Dataset> Size: 14MB
Dimensions:                                     (site: 7, apprate_base: 1,
                                                 dustrad: 10, cec_factor: 10,
                                                 time: 80)
Coordinates:
  * site                                        (site) <U14 392B 'Brazil' ......
  * apprate_base                                (apprate_base) float64 8B 10.0
  * dustrad                                     (dustrad) float64 80B 0.25 .....
  * cec_factor                                  (cec_factor) float64 80B 0.1 ...
  * time                                        (time) float64 640B 0.0 ... 79.0
Data variables: (12/34)
    rco2_correction_factor                      (site, apprate_base, dustrad, cec_factor) float64 6kB ...
    M_GCAM                                      (site, apprate_base, dustrad, cec_factor, time) float64 448kB ...
    mu                                          (site, apprate_base, dustrad, cec_factor, time) float64 448kB ...
    M_new                                       (site, apprate_base, dustrad, cec_factor, time) float64 448kB ...
    A_total_ERW                                 (site, apprate_base, dustrad, cec_factor, time) float64 448kB ...
    cropland_ha_total                           (site, apprate_base, dustrad, cec_factor) float64 6kB ...
    ...                                          ...
    total_dissolution_region                    (site, apprate_base, dustrad, cec_factor, time) float64 448kB ...
    int_dust_ton_ha_yr_region                   (site, apprate_base, dustrad, cec_factor, time) float64 448kB ...
    fraction_total_dissolved_fromIntFlx_region  (site, apprate_base, dustrad, cec_factor, time) float64 448kB ...
    gcam_cdr_globe                              (apprate_base, dustrad, cec_factor, time) float64 64kB ...
    cdr_adv_globe                               (apprate_base, dustrad, cec_factor, time) float64 64kB ...
    M_D_region                                  (site, apprate_base, dustrad, cec_factor, time) float64 448kB ...
Attributes:
    region:   Brazil

## Extra step: application -> dissolution -> CDR lag (cation-budget basis)

Optional diagnostic: for every combination of the swept dims, computes how many years elapse between rock **application**, rock **dissolution**, and **realized CDR** (cation-budget basis), per region and globally.

- `lag_ds` -- per-region lag, dims `(site, ...swept dims...)`.
- `global_lag_ds` -- global lag, dims `(metric, ...swept dims...)`, with `'global_curve_summed'` / `'cdr_weighted_regional_avg'` as variables.

Runs on `ds_sum`, not `ds_all` -- the lag calculation only needs region-level variables (`catbudget_*_region`, `M_GCAM`), not the `(time, deployment)` arrays. Slow for large sweeps.

In [26]:
# --- [ EXTRA STEP: application -> dissolution -> CDR lag, cation-budget basis ]
RCO2 = RCO2_GCAM   # [tCO2/tRock]
PERCENTILES = [0.25, 0.50, 0.75]

lag_ds, global_lag_ds = cfp.compute_lag_table_all_dims(ds_sum, RCO2=RCO2, percentiles=PERCENTILES)
ds_sum = ds_sum.merge(lag_ds).merge(global_lag_ds)

print('lag_ds dims:', dict(lag_ds.sizes))
print('global_lag_ds dims:', dict(global_lag_ds.sizes))
ds_sum

AttributeError: module 'ew_workflows.cdr_fxns_postproc' has no attribute 'compute_lag_table_all_dims'

## Save the result
Save to the same postproc folder that we read the data in from.

In [27]:
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++
# --- save result
SAVE_FN_ALL = "ds_cdr_globalScaling_constApp_ALL.zarr"
SAVE_FN_SUM = "ds_cdr_globalScaling_constApp_SUM.zarr"
ds_all.to_zarr(os.path.join(datpath, SAVE_FN_ALL), zarr_format=2)
ds_sum.to_zarr(os.path.join(datpath, SAVE_FN_SUM), zarr_format=2)
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++

In [24]:
# ---


's3://carbonplan-carbon-removal/ew-workflows-data/scepter/postprocessed/gcam/gcam_opr_psdfull_3axis4x/exe_noWeathering1p5_001'